<a href="https://colab.research.google.com/github/Karna2006/RAG-Basics/blob/main/Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pymupdf sentence-transformers langchain \
            langchain-text-splitters langchain-community \
            langchain-core chromadb -q

# Note: we do NOT install langchain-openai
# We wire in Gemini + sentence-transformers manually
# Everything else (LCEL, Chroma, prompts) is pure LangChain

import fitz, numpy as np
import google.generativeai as genai
from google.colab import userdata, files

from langchain_text_splitters   import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents   import Document
from langchain_core.prompts     import ChatPromptTemplate
from langchain_core.runnables   import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers      import SentenceTransformer

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Imports done.")

Imports done.


In [6]:
#STEP 1 — custom embedding wrapper (bridges sentence-transformers → LangChain)
from langchain_core.embeddings import Embeddings
from typing import List

# LangChain expects an object with .embed_documents() and .embed_query()
# sentence-transformers has .encode() — so we wrap it in one small class.
# This is the ONLY glue code needed to use any embedding model with LangChain.
class STEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, show_progress_bar=True).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text])[0].tolist()

embedding_fn = STEmbeddings()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
#STEP 2 — custom LLM wrapper (bridges Gemini → LangChain)
from langchain_core.language_models import BaseLLM
from langchain_core.outputs import LLMResult, Generation

class GeminiLLM(BaseLLM):
    model_name: str = "gemini-2.5-flash" # Updated model name

    def _generate(self, prompts, stop=None, **kwargs):
        model = genai.GenerativeModel(self.model_name)
        generations = []
        for prompt in prompts:
            resp = model.generate_content(prompt)
            generations.append([Generation(text=resp.text)])
        return LLMResult(generations=generations)

    @property
    def _llm_type(self) -> str:
        return "gemini"

llm = GeminiLLM()

In [12]:
for m in genai.list_models():
    print(f"{m.name}: {m.supported_generation_methods}")

models/gemini-2.5-flash: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts: ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts: ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it: ['generateContent', 'countTokens']
models/gemma-3-4b-it: ['generateContent', 'countTokens']
models/gemma-3-12b-it: ['generateContent', 'countTokens']
mode

In [14]:
#STEP 3 — load PDF + chunk + ingest into Chroma
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Load raw text
doc  = fitz.open(pdf_path)
text = "\n".join(p.get_text() for p in doc)

# LangChain splitter — same RecursiveCharacterTextSplitter you know
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_text(text)

# Wrap chunks in LangChain Document objects (adds metadata support)
docs = [Document(page_content=c, metadata={"source": pdf_path, "chunk_id": i})
        for i, c in enumerate(chunks)]

# Chroma ingests, embeds, and persists in one call
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_fn,
    persist_directory="./chroma_db",   # saves to disk in Colab
    collection_name="my_rag_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"{len(docs)} chunks stored in Chroma.")

Saving Rajkumar_Resume.pdf to Rajkumar_Resume (1).pdf


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

26 chunks stored in Chroma.


In [15]:
#THE LCEL CHAIN — this is the payoff
# The prompt template — same logic as your naive prompt,
# but now it's a reusable, inspectable, composable object
prompt = ChatPromptTemplate.from_template("""
Answer using ONLY the context below.
If the answer is not in the context, say "I don't have that info."

Context:
{context}

Question: {question}
Answer:""")

# Helper: formats retrieved Document objects into one string
# In naive RAG you did: "\n\n---\n\n".join(chunks)
# This does the same thing but for LangChain Document objects
def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

# ── THE LCEL CHAIN ─────────────────────────────────────────────
#
#   {"context": retriever | format_docs,    ← parallel branch
#    "question": RunnablePassthrough()}     ← passthrough branch
#   | prompt                                ← fills template
#   | llm                                  ← generates answer
#   | StrOutputParser()                    ← extracts text
#
# The | operator is LCEL's pipe. Each step receives the output
# of the previous step. The dict at the start runs TWO things
# in parallel: retriever gets the query, passthrough keeps it.

chain = (
    {
        "context":  retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# That's the entire RAG pipeline. 8 lines.

In [16]:
answer = chain.invoke("What is the experience of this candidate?")
print(answer)

25+ years of experience running high-stakes pharmaceutical (API).


In [19]:
#STREAMING — tokens print as they generate (impossible in naive RAG)
for chunk in chain.stream("Summarise the skills of the candidate."):
    print(chunk, end="", flush=True)
print()  # newline at end

The candidate's skills include:
*   Plant & Operations Management
*   End-to-End Program Execution
*   Global Regulatory Audits (US-FDA, MHRA, PMDA, Health Canada)
*   Quality Management Systems (QMS / GMP)
*   Risk Assessment & Mitigation (FMEA)
*   Cross-Functional Leadership (40+ Members)


In [20]:
#BATCH — run multiple questions in one call
questions = [
    "What is this document about?",
    "Who are the roles which the candidate is fit for?",
    "What are the qualities of the candidate?",
]
answers = chain.batch(questions)
for q, a in zip(questions, answers):
    print(f"Q: {q}\nA: {a}\n")

Q: What is this document about?
A: I don't have that info.

Q: Who are the roles which the candidate is fit for?
A: I don't have that info.

Q: What are the qualities of the candidate?
A: The candidate's qualities include: Plant & Operations Management, End-to-End Program Execution, Global Regulatory Audits (US-FDA, MHRA, PMDA, Health Canada), Quality Management Systems (QMS / GMP), Risk Assessment & Mitigation (FMEA), and Cross-Functional Leadership (40+ Members).



In [21]:
#INSPECT RETRIEVED CHUNKS (debugging — add this when answers are wrong)
# Build a retrieval-only chain to see exactly what was fetched
retrieval_chain = (
    retriever
    | RunnableLambda(lambda docs: [
        {"score_rank": i+1, "text": d.page_content[:120], "meta": d.metadata}
        for i, d in enumerate(docs)
    ])
)
chunks_retrieved = retrieval_chain.invoke("What is the refund policy?")
for c in chunks_retrieved:
    print(f"[{c['score_rank']}] {c['text']}...")

# If the retrieved chunks look wrong → go back and tune chunk_size
# If chunks look right but answer is wrong → tune the prompt

[1] structured technical training, and proactive hazard elimination — resulting in zero LTIs (Lost Time Incidents)
over the ...
[2] structured technical training, and proactive hazard elimination — resulting in zero LTIs (Lost Time Incidents)
over the ...
[3] • Directed plant-wide installation of sprinkler and gas suppression systems across manufacturing and storage areas.
Prog...


In [22]:
#RELOAD FROM DISK (don't re-embed every session)
# Next time you open Colab, load the persisted Chroma DB
# instead of re-embedding your entire PDF — saves time + API calls
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_fn,
    collection_name="my_rag_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
# Then rebuild the chain with the new retriever — same 8 lines as before

/tmp/ipykernel_5602/962052013.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(
